# 07 — Đồng bộ HDFS, khởi tạo Hive và truy vấn thực
Chạy notebook 02–03 trước. Bước đầu ghi đè bản sao curated v2 trên HDFS từ dữ liệu batch hiện tại. DDL tái tạo metadata external table theo schema v2, giữ file dữ liệu. Hive dùng MapReduce local cho môi trường demo, chưa phải cụm thực thi Hive phân tán. Lỗi được hiển thị trực tiếp.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
from spark_common import create_spark_session
from project_config import DATA_DIR, RESULTS_DIR, OUTPUT_DIR
from batch_pipeline import sync_hdfs
from hive_client import initialize_hive, query_hive, refresh_historical_partitions, refresh_stream_partitions
spark = create_spark_session("ArXivHiveSync")
try:
    print(sync_hdfs(spark))
finally:
    spark.stop()
initialize_hive()
refresh_historical_partitions()


In [ ]:
display(query_hive("SELECT year, COUNT(*) AS paper_count FROM arxiv_historical GROUP BY year ORDER BY year"))
display(query_hive("SELECT main_group, COUNT(*) AS paper_count FROM arxiv_historical GROUP BY main_group"))
display(query_hive("SELECT year, word, papers_per_1000 FROM hot_keyword_trends ORDER BY year, word"))


Sau khi consumer đã ghi ít nhất một batch, chạy cell dưới để cập nhật partition và kiểm tra số sự kiện. Nếu chưa có dữ liệu stream, Hive có thể trả lỗi đường dẫn chưa tồn tại.

In [ ]:
refresh_stream_partitions()
display(query_hive("SELECT COUNT(*) AS events, COUNT(DISTINCT id) AS distinct_papers FROM arxiv_stream"))
